In [1]:
from oqd_core.interface.digital import *
from oqd_core.compiler.digital.passes import resolve_declarations

In [2]:
# Declare a 5-qubit register `q` and a 5-bit classical register `c`,
# Use `QuantumRef` to reference individual qubits in the gate sequence.

circuit = DigitalCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=5),
        ClassicalDeclaration(name="c", size=5),
    ],
    sequence=[
        Gate(name="h", qreg=QuantumRef(name="q", index=0)),
        Gate(name="h", qreg=QuantumRef(name="q", index=1)),
    ]
)

print(circuit.qasm)

OPENQASM 2.0;
include "qelib1.inc";;
qreg q[5];
creg c[5];
h q[0];
h q[1];



In [3]:
resolved = resolve_declarations(circuit)
resolved

DigitalCircuit(qreg=[], creg=[], declarations=[QuantumDeclaration(name='q', size=5), ClassicalDeclaration(name='c', size=5)], sequence=[Gate(name='h', qreg=QuantumBit(id='q', index=0), creg=None, params=None), Gate(name='h', qreg=QuantumBit(id='q', index=1), creg=None, params=None)])

In [4]:
# Use `AliasDeclaration` to create a named sub-register
# `ancilla` aliases `q[3:5]`
# `QuantumRef(name="ancilla", index=0)` resolves to `q[3]`

circuit = DigitalCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=5),
        AliasDeclaration(name="ancilla", target=QuantumRef(name="q"), begin=3, end=5),
    ],
    sequence=[
        Gate(name="h", qreg=QuantumRef(name="q", index=0)),
        Gate(name="h", qreg=QuantumRef(name="ancilla", index=0)),
        Gate(name="h", qreg=QuantumRef(name="ancilla", index=1)),
    ]
)

print(circuit.qasm)

OPENQASM 2.0;
include "qelib1.inc";;
qreg q[5];
let ancilla = q[3:5];
h q[0];
h ancilla[0];
h ancilla[1];



In [5]:
resolved = resolve_declarations(circuit)

for op in resolved.sequence:
    print(f"{op.name}: {op.qreg}")

h: id='q' index=0
h: id='q' index=3
h: id='q' index=4


In [6]:
# The `qreg`/`creg` and `declarations` entry points are independent

q = QuantumRegister(id="r", reg=3)

circuit = DigitalCircuit(
    qreg=[q],
    declarations=[
        QuantumDeclaration(name="q", size=2),
    ],
    sequence=[
        Gate(name="h", qreg=q[0]),
        Gate(name="h", qreg=QuantumRef(name="q", index=0)),
    ]
)

resolved = resolve_declarations(circuit)

for op in resolved.sequence:
    print(f"{op.name}: {op.qreg}")

h: id='r' index=0
h: id='q' index=0
